In [2]:
import torch
from transformers import AutoTokenizer
from qwen3_5_4b import Qwen3_5Model


In [3]:
model = Qwen3_5Model.from_pretrained(
    "Qwen/Qwen3.5-4B",
    device="cuda:7",
    dtype=torch.bfloat16,
)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [4]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-4B")

special_tokens_dict = {
    "context": tokenizer.convert_tokens_to_ids("<|fim_prefix|>"),
    "question": tokenizer.convert_tokens_to_ids("<|fim_middle|>"),
    "decide": tokenizer.convert_tokens_to_ids("<|fim_suffix|>"),
    "option_start": tokenizer.convert_tokens_to_ids("<|box_start|>"),
    "option_end": tokenizer.convert_tokens_to_ids("<|box_end|>"),
}

inv_special_tokens_dict = {v: k for k, v in special_tokens_dict.items()}

test_message = """<|fim_prefix|>I was charged twice.
<|fim_middle|>Which department should handle this?
<|box_start|>billing<|box_end|>
<|box_start|>shipping<|box_end|>
<|box_start|>returns<|box_end|>
<|fim_suffix|>"""

token_ids = tokenizer(test_message, return_tensors="pt").to(next(model.parameters()).device)["input_ids"]

context_token_pos = (token_ids == special_tokens_dict["context"]).nonzero(as_tuple=True)[1].item()
question_token_pos = (token_ids == special_tokens_dict["question"]).nonzero(as_tuple=True)[1].item()
decide_token_pos = (token_ids == special_tokens_dict["decide"]).nonzero(as_tuple=True)[1].item()
option_start_token_positions = (token_ids == special_tokens_dict["option_start"]).nonzero(as_tuple=True)[1].tolist()
option_end_token_positions = (token_ids == special_tokens_dict["option_end"]).nonzero(as_tuple=True)[1].tolist()

print("=== Encoded Input ===")
print(token_ids)


=== Encoded Input ===
tensor([[248060,     40,    557,  11102,  10598,     13,    198, 248061,  22365,
           9016,   1220,   3579,    411,     30,    198, 248049,  37355, 248050,
            198, 248049,  25660, 248050,    198, 248049,   4077, 248050,    198,
         248062]], device='cuda:7')


In [ ]:
class PointerHead(torch.nn.Module):
    def __init__(self, hidden_dim=2560, pointer_dim=256):
        super().__init__()
        self.query = torch.nn.Linear(hidden_dim, pointer_dim)
        self.key = torch.nn.Linear(hidden_dim, pointer_dim)
        self.scale = pointer_dim ** -0.5

    def forward(self, decide, options):
        return (self.key(options) @ self.query(decide)) * self.scale

In [ ]:
with torch.inference_mode():
    # We assume BS 1
    hidden_state = model.partial_forward(token_ids)[0]
print(hidden_state.shape)

context_state = hidden_state[context_token_pos]
question_state = hidden_state[question_token_pos]
decide_state = hidden_state[decide_token_pos]
option_start_states = [hidden_state[pos] for pos in option_start_token_positions]
option_end_states = [hidden_state[pos] for pos in option_end_token_positions]


torch.Size([28, 2560])
torch.Size([2560]) torch.Size([2560]) torch.Size([2560])
